In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "src" / "database").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Не найдена папка src/database")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from sqlalchemy import inspect, text
from IPython.display import display
from database.connection import engine
from database.connection import engine, SessionLocal

inspector = inspect(engine)
table_names = inspector.get_table_names()

print(f"Connection success. Tables: {len(table_names)}")
print(table_names)

Connection success. Tables: 13
['datasets', 'annotations', 'dataset_classes', 'sequences', 'benchmark_trackers', 'detection_sets', 'evaluation_metrics', 'tracking_results', 'evaluation_runs', 'experiments', 'detections', 'benchmark_results', 'trackers']


In [3]:
# with engine.begin() as connection:
#     connection.execute(text("""
#         ALTER TABLE benchmark_results
#         ALTER COLUMN metric_value DROP NOT NULL;
#     """))

# print("Column metric_value now accepts NULL.")

# inspector = inspect(engine)

# columns = inspector.get_columns("benchmark_results")
# metric_column = next(
#     col for col in columns if col["name"] == "metric_value"
# )

# display({
#     "column": "metric_value",
#     "nullable": metric_column["nullable"],
#     "type": str(metric_column["type"]),
# })

In [4]:
# with engine.begin() as connection:
#     connection.execute(text("""
#         CREATE TABLE benchmark_trackers (
#             id SERIAL PRIMARY KEY,
#             source_key VARCHAR(200) NOT NULL UNIQUE,
#             name VARCHAR(100) NOT NULL,
#             family VARCHAR(100),
#             description TEXT,
#             repository_url VARCHAR(1000),
#             source_metadata JSONB
#         );

#         CREATE INDEX ix_benchmark_trackers_name
#         ON benchmark_trackers (name);

#         ALTER TABLE benchmark_results
#         DROP CONSTRAINT benchmark_results_tracker_id_fkey;

#         ALTER TABLE benchmark_results
#         DROP COLUMN tracker_id;

#         ALTER TABLE benchmark_results
#         ADD COLUMN benchmark_tracker_id INTEGER NOT NULL
#         REFERENCES benchmark_trackers(id)
#         ON DELETE RESTRICT;

#         CREATE INDEX ix_benchmark_results_benchmark_tracker_id
#         ON benchmark_results (benchmark_tracker_id);
#     """))

# print("Benchmark schema updated successfully.")

In [5]:
inspector = inspect(engine)

print("Benchmark trackers:")
for col in inspector.get_columns("benchmark_trackers"):
    print(col["name"], col["type"], col["nullable"])

print("\nBenchmark results:")
for col in inspector.get_columns("benchmark_results"):
    print(col["name"], col["type"], col["nullable"])

print("\nForeign keys:")
for fk in inspector.get_foreign_keys("benchmark_results"):
    print(fk["constrained_columns"], "->", fk["referred_table"])

Benchmark trackers:
id INTEGER False
source_key VARCHAR(200) False
name VARCHAR(100) False
family VARCHAR(100) True
description TEXT True
repository_url VARCHAR(1000) True
source_metadata JSONB True

Benchmark results:
id INTEGER False
benchmark_name VARCHAR(100) False
dataset_name VARCHAR(100) False
metric_name VARCHAR(50) False
metric_value DOUBLE PRECISION True
rank INTEGER True
source_url VARCHAR(1000) True
scraped_at TIMESTAMP False
raw_data JSONB True
benchmark_tracker_id INTEGER False

Foreign keys:
['benchmark_tracker_id'] -> benchmark_trackers


In [6]:
from sqlalchemy import func, select
from database.models.benchmark_tracker import BenchmarkTracker
from database.models.benchmark_result import BenchmarkResult

with SessionLocal() as session:
    tracker_count = session.scalar(
        select(func.count()).select_from(BenchmarkTracker)
    )
    result_count = session.scalar(
        select(func.count()).select_from(BenchmarkResult)
    )
    linked_count = session.scalar(
        select(func.count())
        .select_from(BenchmarkResult)
        .join(
            BenchmarkTracker,
            BenchmarkResult.benchmark_tracker_id
            == BenchmarkTracker.id,
        )
    )

    print(f"Benchmark trackers: {tracker_count:,}")
    print(f"Benchmark results: {result_count:,}")
    print(f"Linked results: {linked_count:,}")

Benchmark trackers: 363
Benchmark results: 107,038
Linked results: 107,038


In [7]:
schema_rows = []

for table_name in table_names:
    for column in inspector.get_columns(table_name):
        schema_rows.append({
            "table": table_name,
            "column": column["name"],
            "type": str(column["type"]),
            "nullable": column["nullable"],
            "primary_key": column.get("primary_key", False),
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

,table,column,type,nullable,primary_key
0,datasets,id,INTEGER,False,False
1,datasets,name,VARCHAR(100),False,False
2,datasets,source,VARCHAR(255),False,False
3,datasets,version,VARCHAR(50),True,False
4,datasets,license,VARCHAR(100),True,False
...,...,...,...,...,...
94,trackers,family,VARCHAR(100),True,False
95,trackers,version,VARCHAR(50),True,False
96,trackers,description,VARCHAR,True,False
97,trackers,repository_url,VARCHAR(500),True,False


In [8]:
foreign_keys = []

for table_name in table_names:
    for fk in inspector.get_foreign_keys(table_name):
        foreign_keys.append({
            "table": table_name,
            "column": ", ".join(fk["constrained_columns"]),
            "references_table": fk["referred_table"],
            "references_column": ", ".join(fk["referred_columns"]),
        })

display(pd.DataFrame(foreign_keys))

,table,column,references_table,references_column
0,annotations,sequence_id,sequences,id
1,dataset_classes,dataset_id,datasets,id
2,sequences,dataset_id,datasets,id
3,detection_sets,sequence_id,sequences,id
4,evaluation_metrics,evaluation_run_id,evaluation_runs,id
5,tracking_results,experiment_id,experiments,id
6,tracking_results,sequence_id,sequences,id
7,evaluation_runs,experiment_id,experiments,id
8,experiments,dataset_id,datasets,id
9,experiments,tracker_id,trackers,id


In [9]:
from loguru import logger
from sqlalchemy import inspect, text

inspector = inspect(engine)

# Проверка внешних ключей
for table_name in inspector.get_table_names():
    foreign_keys = inspector.get_foreign_keys(table_name)

    if foreign_keys:
        for fk in foreign_keys:
            logger.info(
                "{}: {} -> {}.{}",
                table_name,
                fk["constrained_columns"],
                fk["referred_table"],
                fk["referred_columns"],
            )

# Проверка количества записей
with engine.connect() as connection:
    for table_name in inspector.get_table_names():
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        )
        count = result.scalar_one()
        logger.info("{}: {} rows", table_name, count)

2026-11-16 17:37:58.218 | INFO     | __main__:<module>:12 - annotations: ['sequence_id'] -> sequences.['id']
2026-11-16 17:37:58.223 | INFO     | __main__:<module>:12 - dataset_classes: ['dataset_id'] -> datasets.['id']
2026-11-16 17:37:58.227 | INFO     | __main__:<module>:12 - sequences: ['dataset_id'] -> datasets.['id']
2026-11-16 17:37:58.236 | INFO     | __main__:<module>:12 - detection_sets: ['sequence_id'] -> sequences.['id']
2026-11-16 17:37:58.243 | INFO     | __main__:<module>:12 - evaluation_metrics: ['evaluation_run_id'] -> evaluation_runs.['id']
2026-11-16 17:37:58.247 | INFO     | __main__:<module>:12 - tracking_results: ['experiment_id'] -> experiments.['id']
2026-11-16 17:37:58.247 | INFO     | __main__:<module>:12 - tracking_results: ['sequence_id'] -> sequences.['id']
2026-11-16 17:37:58.251 | INFO     | __main__:<module>:12 - evaluation_runs: ['experiment_id'] -> experiments.['id']
2026-11-16 17:37:58.257 | INFO     | __main__:<module>:12 - experiments: ['dataset_id'

2026-11-16 17:37:58.532 | INFO     | __main__:<module>:27 - annotations: 2281088 rows
2026-11-16 17:37:58.536 | INFO     | __main__:<module>:27 - dataset_classes: 45 rows
2026-11-16 17:37:58.539 | INFO     | __main__:<module>:27 - sequences: 133 rows
2026-11-16 17:37:58.541 | INFO     | __main__:<module>:27 - benchmark_trackers: 363 rows
2026-11-16 17:37:58.543 | INFO     | __main__:<module>:27 - detection_sets: 25 rows
2026-11-16 17:37:58.547 | INFO     | __main__:<module>:27 - evaluation_metrics: 0 rows
2026-11-16 17:37:58.548 | INFO     | __main__:<module>:27 - tracking_results: 0 rows
2026-11-16 17:37:58.552 | INFO     | __main__:<module>:27 - evaluation_runs: 0 rows
2026-11-16 17:37:58.556 | INFO     | __main__:<module>:27 - experiments: 0 rows
2026-11-16 17:37:58.655 | INFO     | __main__:<module>:27 - detections: 891359 rows
2026-11-16 17:37:58.661 | INFO     | __main__:<module>:27 - benchmark_results: 107038 rows
2026-11-16 17:37:58.664 | INFO     | __main__:<module>:27 - track

In [10]:
from sqlalchemy import text
from loguru import logger

queries = {
    "datasets": """
        SELECT id, name, source, version
        FROM datasets
        ORDER BY id
    """,
    "sequences": """
        SELECT
            d.name AS dataset,
            COUNT(s.id) AS sequences
        FROM datasets d
        LEFT JOIN sequences s ON s.dataset_id = d.id
        GROUP BY d.id, d.name
        ORDER BY d.name
    """,
    "annotations": """
        SELECT
            d.name AS dataset,
            COUNT(a.id) AS annotations
        FROM datasets d
        LEFT JOIN sequences s ON s.dataset_id = d.id
        LEFT JOIN annotations a ON a.sequence_id = s.id
        GROUP BY d.id, d.name
        ORDER BY d.name
    """,
    "detections": """
        SELECT
            d.name AS dataset,
            COUNT(det.id) AS detections
        FROM datasets d
        LEFT JOIN sequences s ON s.dataset_id = d.id
        LEFT JOIN detection_sets ds ON ds.sequence_id = s.id
        LEFT JOIN detections det ON det.detection_set_id = ds.id
        GROUP BY d.id, d.name
        ORDER BY d.name
    """
}

with engine.connect() as connection:
    for name, query in queries.items():
        logger.info("=== {} ===", name)
        result = connection.execute(text(query))
        for row in result.mappings():
            logger.info("{}", dict(row))

2026-11-16 17:37:58.675 | INFO     | __main__:<module>:44 - === datasets ===
2026-11-16 17:37:58.678 | INFO     | __main__:<module>:47 - {'id': 2, 'name': 'DanceTrack', 'source': 'DanceTrack: Official dataset', 'version': '1.0'}
2026-11-16 17:37:58.679 | INFO     | __main__:<module>:47 - {'id': 3, 'name': 'KITTI Tracking', 'source': 'https://www.cvlibs.net/datasets/kitti/eval_tracking.php', 'version': 'Tracking benchmark'}
2026-11-16 17:37:58.679 | INFO     | __main__:<module>:47 - {'id': 4, 'name': 'MOT17', 'source': 'MOTChallenge: MOT17', 'version': 'MOT17 train'}
2026-11-16 17:37:58.679 | INFO     | __main__:<module>:47 - {'id': 5, 'name': 'MOT20', 'source': 'Hugging Face: Lekim89/MOT20', 'version': 'MOT20'}
2026-11-16 17:37:58.681 | INFO     | __main__:<module>:47 - {'id': 6, 'name': 'VisDrone-MOT', 'source': 'VisDrone2019-MOT', 'version': '2019'}
2026-11-16 17:37:58.681 | INFO     | __main__:<module>:44 - === sequences ===
2026-11-16 17:37:58.686 | INFO     | __main__:<module>:47 

In [11]:
from IPython.display import Markdown, display

for table_name in table_names:
    display(Markdown(f"### Таблица `{table_name}`"))
    df = pd.read_sql(
        text(f'SELECT * FROM "{table_name}" LIMIT 10'),
        engine,
    )
    display(df)

### Таблица `datasets`

,id,name,source,version,license,format
0,2,DanceTrack,DanceTrack: Official dataset,1.0,None,Parquet
1,3,KITTI Tracking,https://www.cvlibs.net/datasets/kitti/eval_tra...,Tracking benchmark,None,Parquet
2,4,MOT17,MOTChallenge: MOT17,MOT17 train,None,Parquet
3,5,MOT20,Hugging Face: Lekim89/MOT20,MOT20,None,Parquet
4,6,VisDrone-MOT,VisDrone2019-MOT,2019,None,Parquet


### Таблица `annotations`

,id,sequence_id,frame,track_id,bbox_left,bbox_top,bbox_width,bbox_height,class_id,mark,visibility
0,1,66,1,0,219.0,377.0,130.0,193.0,1,True,1.0
1,2,66,2,0,229.0,378.0,117.0,192.0,1,True,1.0
2,3,66,3,0,250.0,378.0,103.0,192.0,1,True,1.0
3,4,66,4,0,260.0,378.0,99.0,191.0,1,True,1.0
4,5,66,5,0,270.0,379.0,92.0,190.0,1,True,1.0
5,6,66,6,0,273.0,376.0,84.0,193.0,1,True,1.0
6,7,66,7,0,277.0,372.0,72.0,200.0,1,True,1.0
7,8,66,8,0,274.0,366.0,79.0,206.0,1,True,1.0
8,9,66,9,0,279.0,364.0,74.0,204.0,1,True,1.0
9,10,66,10,0,283.0,363.0,85.0,204.0,1,True,1.0


### Таблица `dataset_classes`

,id,dataset_id,class_id,name
0,3,3,1,Car
1,4,3,2,Van
2,5,3,3,Truck
3,6,3,4,Pedestrian
4,7,3,5,Person
5,8,3,6,Cyclist
6,9,3,7,Tram
7,10,3,8,Misc
8,11,3,9,DontCare
9,12,6,0,Ignored_region


... (16 outputs left out)

### Таблица `benchmark_results`

,id,benchmark_name,dataset_name,metric_name,metric_value,rank,source_url,scraped_at,raw_data,benchmark_tracker_id
0,1,MOTChallenge,MOT17,MOTA,48.000,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
1,2,MOTChallenge,MOT17,IDF1,49.600,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
2,3,MOTChallenge,MOT17,HOTA,39.200,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
3,4,MOTChallenge,MOT17,MT,402.000,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
4,5,MOTChallenge,MOT17,ML,838.000,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
5,6,MOTChallenge,MOT17,FP,23.199,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
6,7,MOTChallenge,MOT17,FN,265.954,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
7,8,MOTChallenge,MOT17,Rcll,52.900,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
8,9,MOTChallenge,MOT17,Prcn,92.800,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1
9,10,MOTChallenge,MOT17,AssA,39.200,None,https://motchallenge.net/methods/MOT-1052-10.html,2026-11-02 15:13:21.766144+00:00,"{'setting': None, 'category': None, 'code_url'...",1


### Таблица `trackers`

,id,name,family,version,description,repository_url,supports_reid


In [12]:
counts = []

for table_name in table_names:
    query = text(f'SELECT COUNT(*) FROM "{table_name}"')
    count = pd.read_sql(query, engine).iloc[0, 0]
    counts.append({
        "table": table_name,
        "rows": count,
    })

counts_df = pd.DataFrame(counts).sort_values(
    "rows", ascending=False
)
display(counts_df)

,table,rows
1,annotations,2281088
10,detections,891359
11,benchmark_results,107038
4,benchmark_trackers,363
3,sequences,133
2,dataset_classes,45
5,detection_sets,25
0,datasets,5
6,evaluation_metrics,0
8,evaluation_runs,0


In [13]:
df = pd.read_sql(
    text('SELECT * FROM "annotations"'),
    engine,
)
display(df)

,id,sequence_id,frame,track_id,bbox_left,bbox_top,bbox_width,bbox_height,class_id,mark,visibility
0,1,66,1,0,219.0,377.0,130.0,193.0,1,True,1.0
1,2,66,2,0,229.0,378.0,117.0,192.0,1,True,1.0
2,3,66,3,0,250.0,378.0,103.0,192.0,1,True,1.0
3,4,66,4,0,260.0,378.0,99.0,191.0,1,True,1.0
4,5,66,5,0,270.0,379.0,92.0,190.0,1,True,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2281083,2281084,198,159,80,1774.0,368.0,63.0,47.0,10,True,0.0
2281084,2281085,198,160,80,1779.0,369.0,63.0,47.0,10,True,0.0
2281085,2281086,198,161,80,1784.0,370.0,63.0,47.0,10,True,0.0
2281086,2281087,198,162,80,1794.0,371.0,63.0,47.0,10,True,0.0


In [14]:
queries = {
    "annotations": """
        SELECT d.name AS dataset, a.class_id, COUNT(*) AS rows
        FROM annotations a
        JOIN sequences s ON s.id = a.sequence_id
        JOIN datasets d ON d.id = s.dataset_id
        GROUP BY d.name, a.class_id
        ORDER BY d.name, a.class_id
    """,
    "detections": """
        SELECT d.name AS dataset, det.class_id, COUNT(*) AS rows
        FROM detections det
        JOIN detection_sets ds ON ds.id = det.detection_set_id
        JOIN sequences s ON s.id = ds.sequence_id
        JOIN datasets d ON d.id = s.dataset_id
        GROUP BY d.name, det.class_id
        ORDER BY d.name, det.class_id
    """,
}

for table_name, query in queries.items():
    display(Markdown(f"### `{table_name}`"))
    df = pd.read_sql(text(query), engine)
    display(df)

### `annotations`

,dataset,class_id,rows
0,DanceTrack,1,574078
1,KITTI Tracking,1,27300
2,KITTI Tracking,2,3301
3,KITTI Tracking,3,1189
4,KITTI Tracking,4,11470
5,KITTI Tracking,5,676
6,KITTI Tracking,6,1938
7,KITTI Tracking,7,595
8,KITTI Tracking,8,793
9,MOT17,1,112297


### `detections`

,dataset,class_id,rows
0,MOT17,None,230216
1,MOT20,None,661143


In [15]:
query = text("""
    SELECT
        d.name AS dataset,
        a.class_id,
        a.mark,
        COUNT(*) AS rows
    FROM annotations a
    JOIN sequences s ON s.id = a.sequence_id
    JOIN datasets d ON d.id = s.dataset_id
    GROUP BY d.name, a.class_id, a.mark
    ORDER BY d.name, a.class_id, a.mark
""")

display(pd.read_sql(query, engine))

,dataset,class_id,mark,rows
0,DanceTrack,1,True,574078
1,KITTI Tracking,1,True,27300
2,KITTI Tracking,2,True,3301
3,KITTI Tracking,3,True,1189
4,KITTI Tracking,4,True,11470
5,KITTI Tracking,5,True,676
6,KITTI Tracking,6,True,1938
7,KITTI Tracking,7,True,595
8,KITTI Tracking,8,True,793
9,MOT17,1,True,112297


In [16]:
for table_name in table_names:
    print(f"\n### {table_name}")
    for column in inspector.get_columns(table_name):
        print(f"  {column['name']}: {column['type']}")
        


### datasets
  id: INTEGER
  name: VARCHAR(100)
  source: VARCHAR(255)
  version: VARCHAR(50)
  license: VARCHAR(100)
  format: VARCHAR(30)

### annotations
  id: INTEGER
  sequence_id: INTEGER
  frame: INTEGER
  track_id: INTEGER
  bbox_left: DOUBLE PRECISION
  bbox_top: DOUBLE PRECISION
  bbox_width: DOUBLE PRECISION
  bbox_height: DOUBLE PRECISION
  class_id: INTEGER
  mark: BOOLEAN
  visibility: DOUBLE PRECISION

### dataset_classes
  id: INTEGER
  dataset_id: INTEGER
  class_id: INTEGER
  name: VARCHAR(100)

### sequences
  id: INTEGER
  dataset_id: INTEGER
  name: VARCHAR(100)
  fps: DOUBLE PRECISION
  frame_count: INTEGER
  width: INTEGER
  height: INTEGER
  description: VARCHAR(500)
  image_url: VARCHAR(1000)

### benchmark_trackers
  id: INTEGER
  source_key: VARCHAR(200)
  name: VARCHAR(100)
  family: VARCHAR(100)
  description: TEXT
  repository_url: VARCHAR(1000)
  source_metadata: JSONB

### detection_sets
  id: INTEGER
  sequence_id: INTEGER
  source: VARCHAR(100)
  dete

In [20]:
query = text("""
    SELECT
    d.name AS dataset,
    s.id AS sequence_id,
    s.name AS sequence,
    s.fps,
    s.frame_count,
    s.width,
    s.height
FROM sequences s
JOIN datasets d ON d.id = s.dataset_id
WHERE d.name IN ('MOT17', 'MOT20')
ORDER BY d.name, s.name;
""")

class_distribution = pd.read_sql(query, engine)
display(class_distribution)

,dataset,sequence_id,sequence,fps,frame_count,width,height
0,MOT17,181,MOT17-02,30.0,600,1920,1080
1,MOT17,182,MOT17-04,30.0,1050,1920,1080
2,MOT17,183,MOT17-05,14.0,837,640,480
3,MOT17,184,MOT17-09,30.0,525,1920,1080
4,MOT17,185,MOT17-10,30.0,654,1920,1080
5,MOT17,186,MOT17-11,30.0,900,1920,1080
6,MOT17,187,MOT17-13,25.0,750,1920,1080
7,MOT20,188,MOT20-01,25.0,429,1920,1080
8,MOT20,189,MOT20-02,25.0,2782,1920,1080
9,MOT20,190,MOT20-03,25.0,2405,1173,880
